In [1]:
print("nfgnjg")

nfgnjg


In [1]:
%pwd
import os
os.chdir("../")
#Just went one step bcakword with this directory....

%pwd

'd:\\mlops\\Crypto_Guardian'

In [6]:
from dataclasses import dataclass
from pathlib import Path

@dataclass
class ModelEvaluationConfig:
    root_dir: Path
    test_data_path: Path 
    model_path: Path
    vetorizer_path: Path
    test_embedding: Path
    evaluation_metrics_path : Path
    curve_img: Path
    mlflow_tracking_uri: str
    mlflow_experiment_name: str
    mlflow_registered_model_name :str

In [ ]:
sahilkumarrock8084

29bedcf08ad331f91cab5cb026e8f07c5075ebaa


In [9]:

from src.Crypto.constants import *
from src.Crypto.utils.helper import read_yaml,create_directories,save_json
from src.Crypto import logger

class ConfigurationManager:
    def __init__(self,config_file_path=CONFIG_FILE_PATH,
                      params_file_path = PARAMS_FILE_PATH,
                      schema_file_path = SCHEMA_FILE_PATH):
        self.config =read_yaml( config_file_path)
        self.parmas = read_yaml(params_file_path)
        self.schema = read_yaml(schema_file_path)
        
        # create_directories([self.config.artifacts_root])
        logger.debug(f"Till Now all the Yaml Files are Read Sucessfully...✅")
    def get_model_evaluation(self)->ModelEvaluationConfig:
        config = self.config.model_evaluation
        mlflow_config = self.config.mlflow
        parmas = self.parmas.SVC
        create_directories([config.root_dir])
        
        model_evaluation_config = ModelEvaluationConfig(
            root_dir=config.root_dir,
            test_data_path=config.test_data_path,
            model_path=config.model_path,
            test_embedding= config.test_embedding,
            vectorizer_path = config.vectorizer_path,
            mlflow_tracking_uri=mlflow_config.mlflow_tracking_uri,
            mlflow_experiment_name=mlflow_config.mlflow_experiment_name,
            mlflow_registered_model_name=mlflow_config.mlflow_registered_model_name,
            evaluation_metrics_path=config.evaluation_metrics_path,
            curve_img= config.curve_img,
            all_parmas = parmas
            # f1_score_threshold=config.f1_score_threshold
        )
        logger.debug("Model Evaluation is working compeletely fine...✅")
        return model_evaluation_config



In [ ]:

import numpy as np
import pickle
import json
import mlflow
import mlflow.sklearn
from mlflow.tracking import MlflowClient
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix
from src.Crypto.entity.config_entity import ModelEvaluationConfig
from src.Crypto import logger
import pandas as pd
import joblib
from sklearn.pipeline import Pipeline
import matplotlib.pyplot as plt
from sklearn.metrics import RocCurveDisplay, PrecisionRecallDisplay, roc_auc_score
import os

class ModelEvaluationComponent:
    def __init__(self,config: ModelEvaluationConfig):
        self.config = config
        self.x_test_emb = np.load(self.config.test_embedding) # <--- Test embeddings bhi chahiye!
        self.test_df = pd.read_csv(self.config.test_data_path)
        self.y_test = self.test_df['Label']
        
        self.vectorizer = joblib.load(self.config.vectorizer_path)

        self.model = joblib.load(self.config.model_path)
        
    def evaluate(self):
        
        self.y_pred = self.model.predict(self.x_test_emb)
        
        self.y_score = self.model.predict_proba(self.x_test_emb)[:, 1] 
        # Calculate metrics
        logger.info("Evaluation metrics calculate kar rahe hain...")
        accuracy = accuracy_score(self.y_test, self.y_pred)
        precision = precision_score(self.y_test, self.y_pred, average='weighted')
        recall = recall_score(self.y_test, self.y_pred, average='weighted')
        f1 = f1_score(self.y_test, self.y_pred, average='weighted')
        roc_auc = roc_auc_score(self.y_test, self.y_score) # ROC Score
        
        # 3. Plots Generation
        # --- ROC Curve ---
        plt.figure(figsize=(10, 6))
        RocCurveDisplay.from_estimator(self.model, self.x_test_emb, self.y_test)
        plt.title("ROC-AUC Curve")
        roc_plot_path = "artifacts\model_evaluation"
        plt.savefig(roc_plot_path)
        plt.close()

        # --- Precision-Recall Curve ---
        plt.figure(figsize=(10, 6))
        PrecisionRecallDisplay.from_estimator(self.model, self.x_test_emb, self.y_test)
        plt.title("Precision-Recall Curve")
        pr_plot_path = "pr_curve.png"
        plt.savefig(pr_plot_path)
        plt.close()
        logger.info(f"Accuracy: {accuracy:.4f}")
        logger.info(f"Precision: {precision:.4f}")
        logger.info(f"Recall: {recall:.4f}")
        logger.info(f"F1 Score: {f1:.4f}")
        
        # Classification report
        class_report = classification_report(self.y_test, self.y_pred)
        logger.info(f"\nClassification Report:\n{class_report}")
        
        # Confusion matrix
        conf_matrix = confusion_matrix(self.y_test, self.y_pred)
        logger.info(f"\nConfusion Matrix:\n{conf_matrix}")
        
        
        
        # Save evaluation metrics locally
        metrics = {
            "accuracy": float(accuracy),
            "precision": float(precision),
            "recall": float(recall),
            "f1_score": float(f1),
            "classification_report": class_report,
            "confusion_matrix": conf_matrix.tolist()
        }
        save_json(Path(self.config.evaluation_metrics_path),metrics)
        logger.info(f"Metrics save kiye: {self.config.evaluation_metrics_path}")
        
        #Starting the MLFLOW Tracking URI
        
        logger.info("MLFLOW Trackign URI is Started...")
        mlflow.set_tracking_uri(self.config.mlflow_tracking_uri)
        mlflow.set_experiment(self.config.mlflow_experiment_name)
        logger.info("MLFLOW Tracking URI Hasbeen Set Sucessfully....")
        
        with mlflow.start_run() as run:
            
            logger.info(f"MLflow run start hua: {run.info.run_id}")
            
            # Log Parameters
            mlflow.log_params(self.config.all_params)
            
            mlflow.log_metric("accuracy", accuracy)
            mlflow.log_metric("precision", precision)
            mlflow.log_metric("recall", recall)
            mlflow.log_metric("f1_score", f1)
            mlflow.log_metric("roc_auc", roc_auc)         
               
            #Creating the Pipeline to log the Model With the Vectorizer
            logger.info(f"Vectorizer load kar rahe hain: {self.config.vectorizer_path}")
            
            mlflow.log_artifact(roc_plot_path, "roc_plots")
            mlflow.log_artifact(pr_plot_path, "pr_plots")
            
            pipeline = Pipeline([
                ('vectorizer',self.vectorizer),
                ('classifier',self.model)
            ])
            #Hum pipeline laga ke hi model ko ship akr rhe hia taaki alag se na karna pade
            
            #Log the Model
            
            mlflow.sklearn.log_model(
                sk_model=pipeline,
                artifact_path='model',
                registered_model_name=self.config.mlflow_registered_model_name
            )
            #model version
            
            model_uri = f"runs:/{run.info.run_id}/model"
            mv = mlflow.register_model(
                model_uri, self.config.mlflow_registered_model_name
            )
            logger.info(f"Model registered with name: {self.config.mlflow_registered_model_name}, version: {mv.version}")
                
        

